# Three-Way Evaluation: Base vs SFT (Self-Distillation) vs DPO

This notebook runs the full held-out evaluation using **actual model weights**:

| Condition | Base Model | Adapter | Training |
|-----------|-----------|---------|----------|
| Base | Qwen 2.5-7B-Instruct | None | — |
| SFT | Qwen 2.5-7B-Instruct | `role-conditioned-distill-final` | Teacher distillation |
| DPO | Qwen 2.5-7B-Instruct | `role-conditioned-dpo-final` | Preference optimization |

All three conditions share the **same 7B base model**, isolating the training method as the only variable.

**Steps:**
1. Load 7B base → generate Base responses
2. Load DPO LoRA adapter → generate DPO responses
3. Swap to SFT LoRA adapter (same base) → generate SFT responses
4. Send all three to LLM judge (Gemini 2.0 Flash via OpenRouter)
5. Aggregate and save results

**Prerequisites:**
- Run `self_distillation_train_colab.ipynb` first to produce the SFT checkpoint (uses 7B base)
- Run `train_colab.ipynb` first to produce the DPO checkpoint (already in repo)
- Colab T4 GPU runtime
- OpenRouter API key

In [ ]:
import os, sys, gc
os.environ['USE_TF'] = '0'
os.environ['TRANSFORMERS_NO_TF'] = '1'

# Mount Google Drive (Colab)
try:
    from google.colab import drive
    drive.mount('/content/drive')
    print('Google Drive mounted.')
except ModuleNotFoundError:
    print('Not on Colab — skipping Drive mount.')

!{sys.executable} -m pip install -q "transformers>=4.40,<5" "peft>=0.10" "accelerate>=0.28" "bitsandbytes>=0.43" "openai" "sentencepiece" "tiktoken"
print('Dependencies installed.')


In [ ]:
# === CONFIG ===
OPENROUTER_API_KEY = os.environ.get('OPENROUTER_API_KEY', '')

BASE_MODEL = 'Qwen/Qwen2.5-7B-Instruct'
JUDGE_MODEL = 'google/gemini-2.0-flash-001'
MAX_NEW_TOKENS = 300
TEMPERATURE = 0.7

from pathlib import Path
import subprocess

def _repo_root() -> Path:
    if os.environ.get('REPO_ROOT'):
        return Path(os.environ['REPO_ROOT']).resolve()
    # Check Google Drive location first (common Colab setup)
    drive_path = Path('/content/drive/MyDrive/CS288/Final Project')
    if drive_path.is_dir() and (drive_path / 'data').is_dir():
        return drive_path
    cwd = Path.cwd().resolve()
    if (cwd / '.git').is_dir() or (cwd / 'data').is_dir():
        return cwd
    for p in (Path('/content/cs288-final-project'),):
        if (p / 'data').is_dir():
            return p.resolve()
    return cwd

REPO_ROOT = _repo_root()
os.chdir(REPO_ROOT)
print(f'Current working directory: {os.getcwd()}')

HOLDOUT_PATH = REPO_ROOT / 'data' / 'scenarios_holdout.jsonl'
DPO_ADAPTER_DIR = REPO_ROOT / 'checkpoints' / 'role-conditioned-dpo-final'
SFT_ADAPTER_DIR = REPO_ROOT / 'checkpoints' / 'role-conditioned-distill-final'
OUTPUT_PATH = REPO_ROOT / 'data' / 'evaluation_results.jsonl'
SUMMARY_PATH = REPO_ROOT / 'data' / 'evaluation_results_summary.json'

# Auto-unzip SFT checkpoint if directory missing but zip is available
if not SFT_ADAPTER_DIR.exists():
    candidate_zips = [
        REPO_ROOT / 'role_conditioned_self_distillation_lora.zip',
        Path('/content/role_conditioned_self_distillation_lora.zip'),
        Path('/content/drive/MyDrive/role_conditioned_self_distillation_lora.zip'),
    ]
    for zip_path in candidate_zips:
        if zip_path.exists():
            print(f'Found SFT zip at {zip_path}, extracting to {REPO_ROOT}...')
            subprocess.run(['unzip', '-o', str(zip_path), '-d', str(REPO_ROOT)], check=True)
            print('Extraction complete.')
            break
    else:
        print('No SFT zip found in standard locations. Will check if directory exists already.')

assert HOLDOUT_PATH.exists(), f'Missing {HOLDOUT_PATH}'
assert DPO_ADAPTER_DIR.exists(), f'Missing {DPO_ADAPTER_DIR} — run train_colab.ipynb first'

HAS_SFT = SFT_ADAPTER_DIR.exists()
print(f'REPO_ROOT  = {REPO_ROOT}')
print(f'Base model: {BASE_MODEL} (shared by all conditions)')
print(f'DPO adapter: {DPO_ADAPTER_DIR} (exists)')
if HAS_SFT:
    print(f'SFT adapter: {SFT_ADAPTER_DIR} (exists)')
else:
    print(f'SFT adapter: {SFT_ADAPTER_DIR} (NOT FOUND — run self_distillation_train_colab.ipynb first or upload the zip)')
    print('Will run two-way eval (Base vs DPO) only.')
print(f'Evaluation mode: {"three-way (Base vs SFT vs DPO)" if HAS_SFT else "two-way (Base vs DPO)"}')


In [ ]:
# === LOAD SCENARIOS ===
import json

with open(HOLDOUT_PATH) as f:
    scenarios = [json.loads(line) for line in f if line.strip()]
print(f'Loaded {len(scenarios)} holdout scenarios')

PROMPT_TEMPLATE = """You are in the following organizational role:
{role_description}

Interaction context:
{context_description}

Situation: {situation}
Background: {background}

Respond appropriately given your role and the situation."""

def build_prompt(scenario):
    return PROMPT_TEMPLATE.format(
        role_description=scenario['role_description'],
        context_description=scenario['context_description'],
        situation=scenario['situation'],
        background=scenario['background'],
    )

In [ ]:
# === HELPERS ===
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel

has_cuda = torch.cuda.is_available()

def load_model_and_tokenizer(model_name):
    """Load a base model with 4-bit quantization (if GPU available)."""
    print(f'Loading {model_name}...')
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    tokenizer.padding_side = 'right'

    bnb_config = None
    torch_dtype = torch.float32
    if has_cuda:
        bnb_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type='nf4',
            bnb_4bit_use_double_quant=True,
            bnb_4bit_compute_dtype=torch.bfloat16,
        )
        torch_dtype = torch.bfloat16

    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        torch_dtype=torch_dtype,
        device_map='auto',
        quantization_config=bnb_config,
    )
    model.eval()
    if has_cuda:
        print(f'  GPU memory: {torch.cuda.memory_allocated() / 1e9:.1f} GB')
    return model, tokenizer


def unload_model(model):
    """Free GPU memory."""
    del model
    gc.collect()
    if has_cuda:
        torch.cuda.empty_cache()
        print(f'  GPU memory after unload: {torch.cuda.memory_allocated() / 1e9:.1f} GB')


def generate_response(mdl, tok, prompt, max_new_tokens=MAX_NEW_TOKENS, temperature=TEMPERATURE):
    """Generate a single response."""
    inputs = tok(prompt, return_tensors='pt', truncation=True, max_length=1024)
    inputs = {k: v.to(mdl.device) for k, v in inputs.items()}
    with torch.no_grad():
        outputs = mdl.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=temperature,
            do_sample=temperature > 0,
            top_p=0.95,
            pad_token_id=tok.eos_token_id,
        )
    new_tokens = outputs[0][inputs['input_ids'].shape[1]:]
    return tok.decode(new_tokens, skip_special_tokens=True).strip()


def generate_all(mdl, tok, scenarios, label=''):
    """Generate responses for all scenarios."""
    responses = []
    for i, scenario in enumerate(scenarios):
        prompt = build_prompt(scenario)
        resp = generate_response(mdl, tok, prompt)
        responses.append(resp)
        if (i + 1) % 10 == 0:
            print(f'  {label} [{i+1}/{len(scenarios)}] done')
    print(f'  {label}: {len(responses)} responses generated.')
    return responses

print('Helpers ready.')

In [ ]:
# === GENERATE ALL RESPONSES (single model load) ===
# All conditions share the same 7B base — load once, swap LoRA adapters.

model, tokenizer = load_model_and_tokenizer(BASE_MODEL)

print('\n--- Generating BASE responses ---')
base_responses = generate_all(model, tokenizer, scenarios, label='Base')

print(f'\nLoading DPO adapter from {DPO_ADAPTER_DIR}...')
model = PeftModel.from_pretrained(model, str(DPO_ADAPTER_DIR), adapter_name='dpo')
model.eval()
if has_cuda:
    print(f'  GPU memory with DPO adapter: {torch.cuda.memory_allocated() / 1e9:.1f} GB')

print('\n--- Generating DPO responses ---')
dpo_responses = generate_all(model, tokenizer, scenarios, label='DPO')

In [ ]:
# === SFT RESPONSES (same 7B base, swap adapter) ===

sft_responses = None

if HAS_SFT:
    print(f'Loading SFT adapter from {SFT_ADAPTER_DIR}...')
    model.load_adapter(str(SFT_ADAPTER_DIR), adapter_name='sft')
    model.set_adapter('sft')
    model.eval()
    if has_cuda:
        print(f'  GPU memory with both adapters: {torch.cuda.memory_allocated() / 1e9:.1f} GB')

    print('\n--- Generating SFT responses ---')
    sft_responses = generate_all(model, tokenizer, scenarios, label='SFT')
else:
    print('Skipping SFT — no checkpoint found.')
    print('To include SFT, run self_distillation_train_colab.ipynb first.')

print('\nUnloading model...')
unload_model(model)
del tokenizer
gc.collect()

In [ ]:
# === SANITY CHECK: are responses different? ===
import difflib

def check_similarity(resps_a, resps_b, label_a, label_b):
    n_identical = sum(1 for a, b in zip(resps_a, resps_b) if a.strip() == b.strip())
    avg_sim = sum(difflib.SequenceMatcher(None, a, b).ratio() for a, b in zip(resps_a, resps_b)) / len(resps_a)
    print(f'  {label_a} vs {label_b}: {n_identical} identical, {avg_sim:.1%} avg similarity')
    if n_identical == len(resps_a):
        print(f'    WARNING: All responses identical! Adapter may not be loaded.')
    return avg_sim

print('Response similarity check:')
check_similarity(base_responses, dpo_responses, 'Base', 'DPO')
if sft_responses:
    check_similarity(base_responses, sft_responses, 'Base', 'SFT')
    check_similarity(sft_responses, dpo_responses, 'SFT', 'DPO')

# Show one example
print(f'\nExample scenario 0: {scenarios[0]["context"]} ({scenarios[0]["role"]["hierarchy"]}/{scenarios[0]["role"]["power_relation"]})')
print(f'\n--- BASE (first 250 chars) ---\n{base_responses[0][:250]}')
print(f'\n--- DPO (first 250 chars) ---\n{dpo_responses[0][:250]}')
if sft_responses:
    print(f'\n--- SFT (first 250 chars) ---\n{sft_responses[0][:250]}')

In [ ]:
# === LLM JUDGE SETUP ===
import random
from openai import OpenAI
from collections import defaultdict

random.seed(42)
client = OpenAI(api_key=OPENROUTER_API_KEY, base_url='https://openrouter.ai/api/v1')

def strip_json_fences(text):
    text = text.strip()
    if text.startswith('```json'):
        text = text[7:]
    elif text.startswith('```'):
        text = text[3:]
    if text.endswith('```'):
        text = text[:-3]
    return text.strip()


# --- THREE-WAY JUDGE ---
JUDGE_3WAY_PROMPT = """You are an expert organizational behavior evaluator. You will evaluate three responses to the same workplace scenario. You do NOT know which response comes from which system.

## Scenario
**Speaker's role:** {role_description}
**Interaction context:** {context_description}
**Situation:** {situation}
**Background:** {background}

## Response A:
{resp_a}

## Response B:
{resp_b}

## Response C:
{resp_c}

## Evaluation
Rate each response on these dimensions (1-5 scale):
1. **Appropriateness**: Does the tone and content match the speaker's hierarchical position and power relationship?
2. **Role Fidelity**: Does the response reflect the speaker's hierarchy level, functional domain, and seniority?
3. **Collaborative Realism**: Does the response exhibit realistic collaborative characteristics?

Then rank all three from best to worst.

Output as JSON:
{{
  "scores": {{
    "A": {{"appropriateness": <1-5>, "role_fidelity": <1-5>, "collaborative_realism": <1-5>}},
    "B": {{"appropriateness": <1-5>, "role_fidelity": <1-5>, "collaborative_realism": <1-5>}},
    "C": {{"appropriateness": <1-5>, "role_fidelity": <1-5>, "collaborative_realism": <1-5>}}
  }},
  "ranking": ["<best>", "<middle>", "<worst>"],
  "notes": "<brief rationale>"
}}"""

# --- TWO-WAY JUDGE (fallback if no SFT) ---
JUDGE_2WAY_PROMPT = """You are an expert organizational behavior evaluator. You will evaluate two responses to the same workplace scenario. You do NOT know which response comes from which model.

## Scenario
**Speaker's role:** {role_description}
**Interaction context:** {context_description}
**Situation:** {situation}
**Background:** {background}

## Response A:
{response_a}

## Response B:
{response_b}

## Evaluation
Rate each response on these dimensions (1-5 scale):
1. **Appropriateness**: Does the tone and content match the speaker's hierarchical position and power relationship?
2. **Role Fidelity**: Does the response reflect the speaker's hierarchy level, functional domain, and seniority?
3. **Collaborative Realism**: Does the response exhibit realistic collaborative characteristics?

Then determine which is better overall, or if they are tied.

Output as JSON:
{{
    "response_a": {{"appropriateness": <1-5>, "role_fidelity": <1-5>, "collaborative_realism": <1-5>, "rationale": "<brief>"}},
    "response_b": {{"appropriateness": <1-5>, "role_fidelity": <1-5>, "collaborative_realism": <1-5>, "rationale": "<brief>"}},
    "winner": "<A or B or tie>",
    "winner_rationale": "<why>"
}}"""

print(f'Judge model: {JUDGE_MODEL}')
print(f'Evaluation mode: {"three-way" if sft_responses else "two-way"}')

In [ ]:
# === RUN EVALUATION ===

systems = ['base', 'sft', 'dpo'] if sft_responses else ['base', 'dpo']
wins = {s: 0 for s in systems}
wins['tie'] = 0
dim_scores = {s: defaultdict(list) for s in systems}
results = []
errors = []

print(f'Evaluating {len(scenarios)} scenarios ({"three" if sft_responses else "two"}-way)...\n')

for i, scenario in enumerate(scenarios):
    try:
        if sft_responses:
            # --- THREE-WAY ---
            all_resps = {
                'base': base_responses[i],
                'sft': sft_responses[i],
                'dpo': dpo_responses[i],
            }
            # Randomize letter assignment
            mapping = list(all_resps.items())  # [(system, response), ...]
            random.shuffle(mapping)
            letters = ['A', 'B', 'C']
            letter_to_system = {letters[j]: mapping[j][0] for j in range(3)}
            system_to_letter = {v: k for k, v in letter_to_system.items()}

            prompt = JUDGE_3WAY_PROMPT.format(
                role_description=scenario['role_description'],
                context_description=scenario['context_description'],
                situation=scenario['situation'],
                background=scenario['background'],
                resp_a=mapping[0][1],
                resp_b=mapping[1][1],
                resp_c=mapping[2][1],
            )
            raw = client.chat.completions.create(
                model=JUDGE_MODEL,
                messages=[{'role': 'user', 'content': prompt + '\n\nRespond with valid JSON only.'}],
                temperature=0.1,
            ).choices[0].message.content.strip()
            judgment = json.loads(strip_json_fences(raw))

            # Extract ranking
            ranking_letters = judgment.get('ranking', [])
            ranking_systems = [letter_to_system[x] for x in ranking_letters if x in letter_to_system]
            if len(ranking_systems) >= 1:
                wins[ranking_systems[0]] += 1

            # Extract scores
            for letter, score_dict in judgment.get('scores', {}).items():
                if letter in letter_to_system:
                    sys_name = letter_to_system[letter]
                    for dim in ['appropriateness', 'role_fidelity', 'collaborative_realism']:
                        if dim in score_dict:
                            dim_scores[sys_name][dim].append(score_dict[dim])

            record = {
                'scenario_index': i,
                'context': scenario.get('context', ''),
                'hierarchy': scenario.get('role', {}).get('hierarchy', ''),
                'function': scenario.get('role', {}).get('function', ''),
                'power_relation': scenario.get('role', {}).get('power_relation', ''),
                'role_description': scenario['role_description'],
                'responses': all_resps,
                'letter_to_system': letter_to_system,
                'judgment': judgment,
                'ranking_systems': ranking_systems,
            }

        else:
            # --- TWO-WAY ---
            base_resp = base_responses[i]
            dpo_resp = dpo_responses[i]

            if random.random() < 0.5:
                resp_a, resp_b = base_resp, dpo_resp
                a_is = 'base'
            else:
                resp_a, resp_b = dpo_resp, base_resp
                a_is = 'dpo'

            prompt = JUDGE_2WAY_PROMPT.format(
                role_description=scenario['role_description'],
                context_description=scenario['context_description'],
                situation=scenario['situation'],
                background=scenario['background'],
                response_a=resp_a,
                response_b=resp_b,
            )
            raw = client.chat.completions.create(
                model=JUDGE_MODEL,
                messages=[{'role': 'user', 'content': prompt + '\n\nRespond with valid JSON only.'}],
                temperature=0.1,
            ).choices[0].message.content.strip()
            judgment = json.loads(strip_json_fences(raw))

            winner_raw = judgment.get('winner', 'tie').strip().upper()
            if winner_raw == 'A':
                actual_winner = a_is
            elif winner_raw == 'B':
                actual_winner = 'dpo' if a_is == 'base' else 'base'
            else:
                actual_winner = 'tie'
            wins[actual_winner] += 1

            a_scores = judgment.get('response_a', {})
            b_scores = judgment.get('response_b', {})
            base_sc = a_scores if a_is == 'base' else b_scores
            dpo_sc = b_scores if a_is == 'base' else a_scores
            for dim in ['appropriateness', 'role_fidelity', 'collaborative_realism']:
                if dim in base_sc: dim_scores['base'][dim].append(base_sc[dim])
                if dim in dpo_sc: dim_scores['dpo'][dim].append(dpo_sc[dim])

            record = {
                'scenario_index': i,
                'context': scenario.get('context', ''),
                'hierarchy': scenario.get('role', {}).get('hierarchy', ''),
                'function': scenario.get('role', {}).get('function', ''),
                'power_relation': scenario.get('role', {}).get('power_relation', ''),
                'role_description': scenario['role_description'],
                'baseline_response': base_resp,
                'dpo_response': dpo_resp,
                'judge_result': judgment,
                'actual_winner': actual_winner,
                'a_is': a_is,
            }

        results.append(record)
        if (i + 1) % 5 == 0:
            print(f'  [{i+1}/{len(scenarios)}] wins so far: {dict(wins)}')

    except Exception as e:
        print(f'  Error on scenario {i}: {e}')
        errors.append((i, str(e)))

print(f'\nDone. Evaluated: {len(results)}, Errors: {len(errors)}')
print(f'Final wins: {dict(wins)}')

In [ ]:
# === SAVE RESULTS ===

with open(OUTPUT_PATH, 'w') as f:
    for rec in results:
        f.write(json.dumps(rec) + '\n')
print(f'Saved {len(results)} results to {OUTPUT_PATH}')

total = len(results)
summary = {
    'total_scenarios': total,
    'mode': 'three-way' if sft_responses else 'two-way',
    'models': {
        'base': BASE_MODEL,
        'sft': BASE_MODEL + ' + LoRA' if sft_responses else None,
        'dpo': BASE_MODEL + ' + LoRA',
    },
    'wins': wins,
    'win_rates': {k: v / max(total, 1) for k, v in wins.items()},
    'mean_scores': {
        sys_name: {
            dim: sum(vals) / max(len(vals), 1)
            for dim, vals in dims.items()
        }
        for sys_name, dims in dim_scores.items()
    },
}
with open(SUMMARY_PATH, 'w') as f:
    json.dump(summary, f, indent=2)
print(f'Saved summary to {SUMMARY_PATH}')
print(json.dumps(summary, indent=2))

In [ ]:
# === PRINT FULL RESULTS ===

print('=' * 60)
print(f'RESULTS ({total} scenarios, {"three" if sft_responses else "two"}-way)')
print('=' * 60)
for s in systems:
    print(f'  {s:>8} wins: {wins[s]:>3} ({100*wins[s]/max(total,1):.1f}%)')
if 'tie' in wins and wins['tie'] > 0:
    print(f'  {"tie":>8}:      {wins["tie"]:>3} ({100*wins["tie"]/max(total,1):.1f}%)')

print(f'\nMean scores (1-5):')
header = f'  {"Dimension":<25}' + ''.join(f'{s:>10}' for s in systems)
print(header)
print(f'  {"-" * (25 + 10 * len(systems))}')
for dim in ['appropriateness', 'role_fidelity', 'collaborative_realism']:
    row = f'  {dim:<25}'
    for s in systems:
        vals = dim_scores[s][dim]
        m = sum(vals) / max(len(vals), 1)
        row += f'{m:>10.2f}'
    print(row)

# Per-context breakdown
print(f'\nPer-context breakdown:')
ctx_wins = defaultdict(lambda: defaultdict(int))
ctx_avg = defaultdict(lambda: defaultdict(list))
for rec in results:
    ctx = rec['context']
    if sft_responses:
        ranking = rec.get('ranking_systems', [])
        if ranking:
            ctx_wins[ctx][ranking[0]] += 1
        jscores = rec['judgment'].get('scores', {})
        l2s = rec['letter_to_system']
        for letter, sd in jscores.items():
            if letter in l2s:
                avg = sum(sd.get(d, 0) for d in ['appropriateness','role_fidelity','collaborative_realism']) / 3
                ctx_avg[ctx][l2s[letter]].append(avg)
    else:
        ctx_wins[ctx][rec['actual_winner']] += 1
        j = rec['judge_result']
        a_is = rec['a_is']
        bs = j.get('response_a', {}) if a_is == 'base' else j.get('response_b', {})
        ds = j.get('response_b', {}) if a_is == 'base' else j.get('response_a', {})
        ba = sum(bs.get(d, 0) for d in ['appropriateness','role_fidelity','collaborative_realism']) / 3
        da = sum(ds.get(d, 0) for d in ['appropriateness','role_fidelity','collaborative_realism']) / 3
        ctx_avg[ctx]['base'].append(ba)
        ctx_avg[ctx]['dpo'].append(da)

hdr = f'  {"Context":<30} {"n":>3}'
for s in systems:
    hdr += f' {s+" W":>7} {s+" avg":>7}'
print(hdr)
for ctx in sorted(ctx_wins.keys()):
    w = ctx_wins[ctx]
    n = sum(w.values())
    row = f'  {ctx:<30} {n:>3}'
    for s in systems:
        avg = sum(ctx_avg[ctx][s]) / max(len(ctx_avg[ctx][s]), 1) if ctx_avg[ctx][s] else 0
        row += f' {w.get(s, 0):>7} {avg:>7.2f}'
    print(row)

# Per-power-relation breakdown
print(f'\nPer-power-relation breakdown:')
pr_wins = defaultdict(lambda: defaultdict(int))
pr_avg = defaultdict(lambda: defaultdict(list))
for rec in results:
    pr = rec['power_relation']
    if sft_responses:
        ranking = rec.get('ranking_systems', [])
        if ranking:
            pr_wins[pr][ranking[0]] += 1
        jscores = rec['judgment'].get('scores', {})
        l2s = rec['letter_to_system']
        for letter, sd in jscores.items():
            if letter in l2s:
                avg = sum(sd.get(d, 0) for d in ['appropriateness','role_fidelity','collaborative_realism']) / 3
                pr_avg[pr][l2s[letter]].append(avg)
    else:
        pr_wins[pr][rec['actual_winner']] += 1
        j = rec['judge_result']
        a_is = rec['a_is']
        bs = j.get('response_a', {}) if a_is == 'base' else j.get('response_b', {})
        ds = j.get('response_b', {}) if a_is == 'base' else j.get('response_a', {})
        ba = sum(bs.get(d, 0) for d in ['appropriateness','role_fidelity','collaborative_realism']) / 3
        da = sum(ds.get(d, 0) for d in ['appropriateness','role_fidelity','collaborative_realism']) / 3
        pr_avg[pr]['base'].append(ba)
        pr_avg[pr]['dpo'].append(da)

for pr in ['superior', 'subordinate', 'peer', 'cross_functional']:
    if pr in pr_wins:
        w = pr_wins[pr]
        n = sum(w.values())
        row = f'  {pr:<20} {n:>3}'
        for s in systems:
            avg = sum(pr_avg[pr][s]) / max(len(pr_avg[pr][s]), 1) if pr_avg[pr][s] else 0
            row += f' {w.get(s, 0):>7} {avg:>7.2f}'
        print(row)

In [ ]:
# === DOWNLOAD RESULTS ===
try:
    from google.colab import files
    files.download(str(OUTPUT_PATH))
    files.download(str(SUMMARY_PATH))
except ModuleNotFoundError:
    print(f'Not in Colab. Results saved locally:')
    print(f'  {OUTPUT_PATH}')
    print(f'  {SUMMARY_PATH}')

# Ablation: Role-Conditioning Removal

Tests whether the structured role description in the prompt actually matters.
- **With role**: Base model receives the full prompt (role description + context + situation)
- **No role**: Base model receives only the context + situation (role description stripped)

Both are judged against the **full scenario** (the judge still sees the role), so the question is: does telling the model its role improve its response quality?

This runs after the main evaluation — `base_responses` from the earlier cells are reused.

In [ ]:
# === ABLATION: Generate no-role responses ===

NO_ROLE_PROMPT_TEMPLATE = """Interaction context:
{context_description}

Situation: {situation}
Background: {background}

Respond appropriately given the situation."""

def build_no_role_prompt(scenario):
    return NO_ROLE_PROMPT_TEMPLATE.format(
        context_description=scenario['context_description'],
        situation=scenario['situation'],
        background=scenario['background'],
    )

# Reload base model (was freed after main eval)
model_abl, tokenizer_abl = load_model_and_tokenizer(BASE_MODEL)

print('\n--- Generating NO-ROLE responses (base model, no role description) ---')
norole_responses = []
for i, scenario in enumerate(scenarios):
    prompt = build_no_role_prompt(scenario)
    resp = generate_response(model_abl, tokenizer_abl, prompt)
    norole_responses.append(resp)
    if (i + 1) % 10 == 0:
        print(f'  No-Role [{i+1}/{len(scenarios)}] done')
print(f'  No-Role: {len(norole_responses)} responses generated.')

print('\nUnloading model...')
unload_model(model_abl)
del tokenizer_abl
gc.collect()

# Quick check
print(f'\nExample scenario 0:')
print(f'  WITH role (first 200 chars): {base_responses[0][:200]}')
print(f'  NO role   (first 200 chars): {norole_responses[0][:200]}')

In [ ]:
# === ABLATION JUDGE: Base+Role vs Base+NoRole ===

random.seed(123)

abl_wins = {'with_role': 0, 'no_role': 0, 'tie': 0}
abl_dim_scores = {'with_role': defaultdict(list), 'no_role': defaultdict(list)}
abl_results = []
abl_errors = []

print(f'Ablation: judging {len(scenarios)} scenarios (with-role vs no-role)...\n')

for i, scenario in enumerate(scenarios):
    try:
        role_resp = base_responses[i]
        norole_resp = norole_responses[i]

        if random.random() < 0.5:
            resp_a, resp_b = role_resp, norole_resp
            a_is = 'with_role'
        else:
            resp_a, resp_b = norole_resp, role_resp
            a_is = 'no_role'

        prompt = JUDGE_2WAY_PROMPT.format(
            role_description=scenario['role_description'],
            context_description=scenario['context_description'],
            situation=scenario['situation'],
            background=scenario['background'],
            response_a=resp_a,
            response_b=resp_b,
        )
        raw = client.chat.completions.create(
            model=JUDGE_MODEL,
            messages=[{'role': 'user', 'content': prompt + '\n\nRespond with valid JSON only.'}],
            temperature=0.1,
        ).choices[0].message.content.strip()
        judgment = json.loads(strip_json_fences(raw))

        winner_raw = judgment.get('winner', 'tie').strip().upper()
        if winner_raw == 'A':
            actual_winner = a_is
        elif winner_raw == 'B':
            actual_winner = 'no_role' if a_is == 'with_role' else 'with_role'
        else:
            actual_winner = 'tie'
        abl_wins[actual_winner] += 1

        a_scores = judgment.get('response_a', {})
        b_scores = judgment.get('response_b', {})
        wr_sc = a_scores if a_is == 'with_role' else b_scores
        nr_sc = b_scores if a_is == 'with_role' else a_scores
        for dim in ['appropriateness', 'role_fidelity', 'collaborative_realism']:
            if dim in wr_sc: abl_dim_scores['with_role'][dim].append(wr_sc[dim])
            if dim in nr_sc: abl_dim_scores['no_role'][dim].append(nr_sc[dim])

        abl_results.append({
            'scenario_index': i,
            'context': scenario.get('context', ''),
            'power_relation': scenario.get('role', {}).get('power_relation', ''),
            'hierarchy': scenario.get('role', {}).get('hierarchy', ''),
            'with_role_response': role_resp,
            'no_role_response': norole_resp,
            'judge_result': judgment,
            'actual_winner': actual_winner,
            'a_is': a_is,
        })

        if (i + 1) % 10 == 0:
            print(f'  [{i+1}/{len(scenarios)}] with_role={abl_wins["with_role"]} no_role={abl_wins["no_role"]} tie={abl_wins["tie"]}')

    except Exception as e:
        print(f'  Error on scenario {i}: {e}')
        abl_errors.append((i, str(e)))

print(f'\nDone. Errors: {len(abl_errors)}')
print(f'Ablation wins: {dict(abl_wins)}')


In [ ]:
# === ABLATION RESULTS ===

abl_total = sum(abl_wins.values())
print('=' * 60)
print(f'ABLATION: ROLE-CONDITIONING REMOVAL ({abl_total} scenarios)')
print('=' * 60)
print(f'  With role wins: {abl_wins["with_role"]} ({100*abl_wins["with_role"]/max(abl_total,1):.1f}%)')
print(f'  No role wins:   {abl_wins["no_role"]} ({100*abl_wins["no_role"]/max(abl_total,1):.1f}%)')
print(f'  Ties:           {abl_wins["tie"]} ({100*abl_wins["tie"]/max(abl_total,1):.1f}%)')

print(f'\nMean scores (1-5):')
print(f'  {"Dimension":<25} {"With Role":>10} {"No Role":>10} {"Delta":>10}')
print(f'  {"-"*55}')
for dim in ['appropriateness', 'role_fidelity', 'collaborative_realism']:
    wr_mean = sum(abl_dim_scores['with_role'][dim]) / max(len(abl_dim_scores['with_role'][dim]), 1)
    nr_mean = sum(abl_dim_scores['no_role'][dim]) / max(len(abl_dim_scores['no_role'][dim]), 1)
    print(f'  {dim:<25} {wr_mean:>10.2f} {nr_mean:>10.2f} {wr_mean - nr_mean:>+10.2f}')

# Per-power-relation breakdown
print(f'\nPer-power-relation breakdown:')
abl_pr_wins = defaultdict(lambda: defaultdict(int))
for rec in abl_results:
    pr = rec['power_relation']
    abl_pr_wins[pr][rec['actual_winner']] += 1

print(f'  {"Power Relation":<20} {"n":>3} {"Role W":>7} {"NoRole W":>8} {"Tie":>5}')
for pr in ['superior', 'subordinate', 'peer', 'cross_functional']:
    if pr in abl_pr_wins:
        w = abl_pr_wins[pr]
        n = sum(w.values())
        print(f'  {pr:<20} {n:>3} {w.get("with_role",0):>7} {w.get("no_role",0):>8} {w.get("tie",0):>5}')

# Save ablation results
abl_output = REPO_ROOT / 'data' / 'ablation_role_conditioning.jsonl'
abl_summary_path = REPO_ROOT / 'data' / 'ablation_role_conditioning_summary.json'

with open(abl_output, 'w') as f:
    for rec in abl_results:
        f.write(json.dumps(rec) + '\n')

abl_summary = {
    'ablation': 'role_conditioning_removal',
    'total_scenarios': abl_total,
    'wins': abl_wins,
    'win_rates': {k: v / max(abl_total, 1) for k, v in abl_wins.items()},
    'mean_scores': {
        sys_name: {dim: sum(vals) / max(len(vals), 1) for dim, vals in dims.items()}
        for sys_name, dims in abl_dim_scores.items()
    },
}
with open(abl_summary_path, 'w') as f:
    json.dump(abl_summary, f, indent=2)

print(f'\nSaved ablation results to {abl_output}')
print(f'Saved ablation summary to {abl_summary_path}')

# Download
try:
    from google.colab import files
    files.download(str(abl_output))
    files.download(str(abl_summary_path))
except ModuleNotFoundError:
    pass
